In [1]:
!pip install transformers datasets torch accelerate


     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     ------------------ -------------------- 20.5/44.0 kB 59.5 kB/s eta 0:00:01
     ------------------ -------------------- 20.5/44.0 kB 59.5 kB/s eta 0:00:01
     --------------------------- ----------- 30.7/44.0 kB 81.9 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/44.0 kB 103.4 kB/s eta 0:00:01
     --------------------------------------- 44.0/44.0 kB 98.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   -------------------

In [2]:
!pip uninstall -y tensorflow tensorflow-intel keras keras-nightly keras-preprocessing keras-vis


Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: tensorflow_intel 2.18.0
Uninstalling tensorflow_intel-2.18.0:
  Successfully uninstalled tensorflow_intel-2.18.0
Found existing installation: keras 3.7.0
Uninstalling keras-3.7.0:
  Successfully uninstalled keras-3.7.0


You can safely remove it manually.


In [3]:
!pip install torch



In [4]:
!pip install transformers --upgrade

In [2]:
import os
os.environ["USE_TF"] = "0"


In [ ]:
import os
os.environ["USE_TF"] = "0"   # disable TensorFlow

from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# ====================================================
# 1. Load Dataset
# ====================================================
path = r"C:\MAIN\Projects\Sarcasm Detection\Dataset\unique_tweets.csv"
df = pd.read_csv(path)

text_col = [c for c in df.columns if "tweet" in c.lower() or "text" in c.lower()][0]
label_col = [c for c in df.columns if "label" in c.lower()][0]

df = df[[text_col, label_col]]
df.columns = ["text", "label"]

df["label"] = df["label"].map({"YES": 1, "NO": 0})

# ====================================================
# 2. Train-test split
# ====================================================
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# ====================================================
# 3. Load IndicBERT tokenizer + model
# ====================================================
MODEL_NAME = "ai4bharat/indic-bert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ====================================================
# 4. Tokenize
# ====================================================
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["text", "__index_level_0__"])
val_dataset = val_dataset.remove_columns(["text", "__index_level_0__"])

# ====================================================
# 5. Training arguments
# ====================================================
training_args = TrainingArguments(
    output_dir="./indicbert_sarcasm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100
)

# ====================================================
# 6. Trainer
# ====================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ====================================================
# 7. Train
# ====================================================
trainer.train()

# ====================================================
# 8. Save model
# ====================================================
trainer.save_model("./indicbert_finetuned_sarcasm")
tokenizer.save_pretrained("./indicbert_finetuned_sarcasm")

print("Fine-tuned IndicBERT model saved.")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indic-bert.
401 Client Error. (Request ID: Root=1-691d9254-434406b8503defd11a783776;77816eeb-929f-4d57-b3d2-092bb993f8f9)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indic-bert/resolve/main/config.json.
Access to model ai4bharat/indic-bert is restricted. You must have access to it and be authenticated to access it. Please log in.